# TAMPO Algorithms — Colab Test Run Notebook

This notebook tests the currently implemented TAMPO meta-RL algorithms (TAMPO-GCN and TAMPO-LSTM).
Run cells **top to bottom** in order.

## 0. Setup — Clone repo & install dependencies

In [2]:
# ── 0a. Clone the repo ────────────────────────────────────────────
!git clone -b gcn-pyg https://github.com/vikkesh/tampo.git tampo
%cd tampo

Cloning into 'tampo'...
remote: Enumerating objects: 3638, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 3638 (delta 42), reused 68 (delta 23), pack-reused 3541 (from 2)
Receiving objects: 100% (3638/3638), 200.27 MiB | 23.49 MiB/s, done.
Resolving deltas: 100% (71/71), done.
/content/tampo


In [4]:
!nvidia-smi

Thu Jun 11 05:35:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# ── 0b. Install all dependencies ─────────────────────────────────
!pip install -r requirements.txt

In [6]:
# ── 0c. Verify imports ────────────────────────────────────────────
import torch, gym, yaml, json, numpy as np
from torch_geometric.data import Batch, Data
print("torch:", torch.__version__)
print("gym:", gym.__version__)
print("CUDA available:", torch.cuda.is_available())

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


torch: 2.11.0+cu128
gym: 0.26.2
CUDA available: True


## 1. Unit Test — DAGEncoder with Bi-GCN path

Verifies that the encoder:
- Accepts a PyG Batch of **variable-size** graphs
- Produces a context vector of the correct shape 
- Properly handles the bi-directional pass without crashing

In [7]:
import sys, os
sys.path.insert(0, '.')  # tampo root

import torch
from torch_geometric.data import Batch, Data
from algorithms.rl.tampo import DAGEncoder

TASK_FEATURE_DIM = 6
HIDDEN_DIM = 128
SERVER_FEATURE_DIM = 20

encoder = DAGEncoder(
    task_feature_dim=TASK_FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=2,
    encoder_type='gcn',
    server_feature_dim=SERVER_FEATURE_DIM
)
encoder.eval()

# Three graphs with DIFFERENT node counts (8, 12, 20)
graphs = []
for n in [8, 12, 20]:
    x = torch.rand(n, TASK_FEATURE_DIM)
    # Random directed edges (roughly a chain)
    src = torch.arange(0, n - 1)
    dst = torch.arange(1, n)
    edge_index = torch.stack([src, dst], dim=0)
    graphs.append(Data(x=x, edge_index=edge_index, num_nodes=n))

batch = Batch.from_data_list(graphs)

# Dummy task_features tensor
task_features_dummy = torch.zeros(3, 20, TASK_FEATURE_DIM)
server_features = torch.rand(3, SERVER_FEATURE_DIM)

with torch.no_grad():
    encoded_tasks, context = encoder(
        task_features=task_features_dummy,
        graph_batch=batch,
        server_features=server_features
    )

assert context.shape == (3, HIDDEN_DIM * 2), f"Bad context shape: {context.shape}"
print(f"PASS — context shape: {context.shape}  (expected [3, {HIDDEN_DIM*2}])")
print(f"PASS — encoded_tasks shape: {encoded_tasks.shape}")


PASS — context shape: torch.Size([3, 256])  (expected [3, 256])
PASS — encoded_tasks shape: torch.Size([3, 20, 256])


## 2. Generate the Golden Test Dataset

> ⚠️ Run **once only**.  Never re-run between algorithm comparisons.

In [ ]:
!python utils/generate_test_dataset.py --num_dags 20 --output data/test_dags.json

import json
with open("data/test_dags.json") as f:
    ds = json.load(f)
print(f"Dataset contains {len(ds)} entries")
print("First entry keys:", list(ds[0].keys()))

## 3. Quick Training Smoke Test — TAMPO-GCN and TAMPO-LSTM (1 iteration)

Checks that the full training loop executes without shape errors for both algorithms.

In [ ]:
import yaml, sys
sys.path.insert(0, '.')

with open('configs/default_config.yaml') as f:
    full_config = yaml.safe_load(f)

from env.base_offloading_env import TaskOffloadingEnv
from algorithms.rl.tampo import TAMPOFramework

cfg = {}
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(full_config['environment'][sec])

env = TaskOffloadingEnv(cfg)

# Load the generated dataset
import json
with open('data/test_dags.json') as f:
    test_dags = json.load(f)
env.load_task_dataset(test_dags)

for enc_type in ['gcn', 'lstm']:
    print(f'\n======================================')
    print(f'Testing TAMPO-{enc_type.upper()}')
    print(f'======================================')
    tampo_cfg = full_config['algorithms']['tampo'].copy()
    tampo_cfg['encoder_type'] = enc_type
    tampo_cfg['num_meta_iterations'] = 1  # just 1 iteration for smoke test

    framework = TAMPOFramework(env, tampo_cfg)
    results = framework.train(num_meta_iterations=1, num_tasks_per_iter=2)
    
    import os
    os.makedirs('models', exist_ok=True)
    framework.save(f'models/tampo_{enc_type}_checkpoint.pth')
    print(f'SMOKE TEST PASSED for TAMPO-{enc_type.upper()} — checkpoint saved.')


## 4. Run Benchmark on Trained Models

Compares both trained algorithms against the same test dataset.

In [ ]:
import os, sys
sys.path.insert(0, '.')

# Run benchmark for both algorithms
!python benchmark.py \
    --algorithms TAMPO_GCN TAMPO_LSTM \
    --checkpoint_dir models/ \
    --dataset_path data/test_dags.json \
    --output_dir results/


## 5. Download Results

In [ ]:
import os
import glob
from google.colab import files

# Find the most recently created run directory
run_dirs = sorted(glob.glob('results/run_*'), reverse=True)
if run_dirs:
    latest_run = run_dirs[0]
    for fname in ['comparison_bar.png', 'pareto_front.png', 'benchmark_results.csv']:
        path = os.path.join(latest_run, fname)
        if os.path.exists(path):
            files.download(path)
            print(f'Downloaded {path}')
        else:
            print(f'Not found: {path}')
else:
    print('No results found. Run benchmark first.')
